In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from google.colab import files

# 1. Настройка устройства
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")

# 2. Подготовка данных
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

# 3. Новая архитектура 784 -> 32 -> 10
class SimpleMLP(nn.Module):
    def __init__(self):
        super(SimpleMLP, self).__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            # Входной слой (28*28 = 784) -> Скрытый слой (32)
            nn.Linear(28 * 28, 32),
            nn.ReLU(),
            # Скрытый слой (32) -> Выходной слой (10)
            nn.Linear(32, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.layers(x)

model = SimpleMLP().to(device)

# 4. Функция потерь и оптимизатор
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 5. Обучение в FP32
epochs = 5
print("\nНачало обучения (FP32)...")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Эпоха [{epoch+1}/{epochs}], Потери: {running_loss/len(train_loader):.4f}")

# === ПЕРЕВОД МОДЕЛИ В FP16 ===
print("\nКонвертация модели в формат FP16...")
model = model.half()

# 6. Оценка точности в FP16
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device).half()
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Точность модели (FP16): {100 * correct / total:.2f}%\n")

# 7. Сохранение весов: матрица -> bias -> матрица -> bias
bin_filename = "mnist_mlp_32_10_fp16_sorted.bin"

with open(bin_filename, 'wb') as f:
    for name, param in model.named_parameters():
        param_np = param.detach().cpu().numpy()
        f.write(param_np.tobytes())
        print(f" -> {name:<18} | Форма: {str(param_np.shape):<10} | {param_np.nbytes:>6} байт")

print(f"\nВсе параметры записаны в {bin_filename}")

# 8. Скачивание файла на ПК
files.download(bin_filename)

Используемое устройство: cuda


100%|██████████| 9.91M/9.91M [00:00<00:00, 17.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 490kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.49MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.8MB/s]



Начало обучения (FP32)...
Эпоха [1/5], Потери: 0.5092
Эпоха [2/5], Потери: 0.3147
Эпоха [3/5], Потери: 0.2886
Эпоха [4/5], Потери: 0.2652
Эпоха [5/5], Потери: 0.2476

Конвертация модели в формат FP16...
Точность модели (FP16): 93.37%

 -> layers.0.weight    | Форма: (32, 784)  |  50176 байт
 -> layers.0.bias      | Форма: (32,)      |     64 байт
 -> layers.2.weight    | Форма: (10, 32)   |    640 байт
 -> layers.2.bias      | Форма: (10,)      |     20 байт

Все параметры записаны в mnist_mlp_32_10_fp16_sorted.bin


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>